# AGIST attention strength across time

Use the 6,707-cell simulation and its trained model to calculate attention at
each of the four observed times. This is a separate experiment from the
31,816-cell simulation shown in [Figure 2](main_figure_2.ipynb).

The original analysis compares the mean outgoing attention of each cell
with the generator. It uses Spearman correlation among cells with nonzero
strength. Prediction and reference must select the same cells.

Run the cells below from the CytoBridge code environment. The additional
Figure 2 download supplies the observations, model, edge predictor and
generator matrices. No saved predicted attention is used by these cells.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

import CytoBridge as cb
from IPython.display import Image, display

project = Path(os.environ.get("CYTOBRIDGE_PROJECT_DIR", ".")).resolve()
device = os.environ.get("CYTOBRIDGE_DEVICE", "cuda")
run_name = os.environ.get("CYTOBRIDGE_RUN_LABEL", "reader_run")
repo = Path(cb.__file__).resolve().parents[1]

def run(module, *arguments):
    subprocess.run([sys.executable, "-m", module, *map(str, arguments)],
                   cwd=repo, check=True)

figure2_root = Path(os.environ.get("CYTOBRIDGE_AGIST_FIGURE2", project / "data/agist/figure2"))
if not (figure2_root / "attention_recovery/model/model_final").is_file():
    cb.datasets.download("agist", destination=project, kind="agist_figure2_inputs.zip")
inputs = figure2_root / "attention_recovery"
output = project / "outputs" / ("agist_attention_" + run_name)

## Infer attention from the model

At each time, build the graph from all observed cells. Average the absolute
first-layer attention over its eight heads, then calculate each cell's mean
outgoing attention. Save the predicted edges, paired strengths and one
correlation per time.

In [2]:
from reproduction.agist.main_figure import evaluate_attention_recovery

correlations, summary = evaluate_attention_recovery(
    inputs / "mouse_brain_simulation_new.csv", inputs / "model/params.yml",
    inputs / "model", inputs / "mouse_new.pt", inputs / "reference",
    output, device=device)
display(correlations.round({"spearman": 4}))

,time,n_cells,n_compared,spearman
0,0.0,1147,659,0.8672
1,1.0,1619,694,0.7661
2,2.0,1986,1329,0.7314
3,3.0,1955,1419,0.7451


## Summarize the four correlations

Give each time point equal weight. This is the arithmetic mean of four
correlations, not one correlation calculated after pooling all cells.
The original notebook recorded the four values separately. Their mean is
calculated explicitly here.

In [3]:
print(f"Mean Spearman correlation: {summary['mean_spearman']:.5f}")
print(f"Rounded to two decimal places: {summary['mean_spearman']:.2f}")

Mean Spearman correlation: 0.77746
Rounded to two decimal places: 0.78
